# Clearfacts navigation agent test

This notebook is the preferred manual test surface for the experimental `ClearfactsNavigationAgent`.

- It configures the Clearfacts test user source from `cf-integration-tests`
- It can run Playwright MCP in a **visible browser**
- It formats the agent result into readable tables and text blocks instead of raw CLI JSON


In [7]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / "context_db").exists():
        root = root.parent
    if not (root / "context_db").exists():
        raise RuntimeError("Could not locate the cf-ml-context-db repository root from the current notebook session.")
    return root


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from context_db.agents.clearfacts_navigation_agent import ClearfactsNavigationAgent
from context_db.agents.clearfacts_navigation_agent.schemas import ClearfactsNavigationRequest, ClearfactsRole

pd.set_option("display.max_colwidth", 160)
REPO_ROOT


PosixPath('/Users/dirkvangheel/Documents/dev/projects/clearfacts/cf-ml-context-db')

## Runtime configuration

- Set `PLAYWRIGHT_MCP_HEADLESS=false` to follow along in a visible browser
- Set `CLEARFACTS_NAVIGATION_STEP_DELAY_MS` to pause after visible browser actions
- Adjust `PLAYWRIGHT_MCP_ARGS` if you want a specific browser channel such as Chrome


In [8]:
os.environ["CLEARFACTS_TEST_USERS_FILE"] = str(
    (REPO_ROOT.parent / "cf-integration-tests" / "cypress" / "fixtures" / "users.json").resolve()
)
os.environ["PLAYWRIGHT_MCP_COMMAND"] = "npx"
os.environ["PLAYWRIGHT_MCP_ARGS"] = "-y @playwright/mcp@latest --browser chrome"
os.environ["PLAYWRIGHT_MCP_HEADLESS"] = "false"
os.environ["CLEARFACTS_NAVIGATION_STEP_DELAY_MS"] = "1200"
os.environ["DB_CONFIG_FILE"] = str((REPO_ROOT / "config/database.ini").resolve())

runtime_settings = {
    key: os.environ[key]
    for key in [
        "CLEARFACTS_TEST_USERS_FILE",
        "PLAYWRIGHT_MCP_COMMAND",
        "PLAYWRIGHT_MCP_ARGS",
        "PLAYWRIGHT_MCP_HEADLESS",
        "CLEARFACTS_NAVIGATION_STEP_DELAY_MS",
    ]
}
pd.DataFrame([runtime_settings])


,CLEARFACTS_TEST_USERS_FILE,PLAYWRIGHT_MCP_COMMAND,PLAYWRIGHT_MCP_ARGS,PLAYWRIGHT_MCP_HEADLESS,CLEARFACTS_NAVIGATION_STEP_DELAY_MS
0,/Users/dirkvangheel/Documents/dev/projects/clearfacts/cf-integration-tests/cypress/fixtures/users.json,npx,-y @playwright/mcp@latest --browser chrome,false,1200


In [9]:
agent = ClearfactsNavigationAgent()

request = ClearfactsNavigationRequest(
    instruction="Log in as an SME user",
    role=ClearfactsRole.SME_ADMIN,
    include_snapshot=True,
)

display(Markdown("## Request"))
display(pd.DataFrame([request.model_dump(mode="json")]))


## Request

,instruction,role,start_target,max_steps,include_snapshot
0,Log in as an SME user,sme_admin,None,8,True


In [10]:
result = agent.invoke(request)
result.status.value


/Users/dirkvangheel/homebrew/Caskroom/miniforge/base/envs/cf-ml-insights-agent/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ClearfactsNavigationPlan(..., text=None, key=None)]), input_type=ClearfactsNavigationPlan])
  return self.__pydantic_serializer__.to_python(


'completed'

In [11]:
def display_navigation_result(result) -> None:
    summary_lines = [
        "## Summary",
        f"- **Status:** `{result.status.value}`",
        f"- **Message:** {result.message or '_none_'}",
        f"- **Planned steps:** {len(result.plan.steps)}",
        f"- **Executed steps:** {len(result.trace)}",
        f"- **Available tools:** {', '.join(result.tool_inventory) or '_none_'}",
    ]
    display(Markdown("\n".join(summary_lines)))

    display(Markdown("## Planned steps"))
    for index, step in enumerate(result.plan.steps, start=1):
        details = [
            f"### Step {index}: `{step.action_type.value}`",
            step.description,
        ]
        if step.target:
            details.append(f"- **Target:** `{step.target.value}`")
        if step.selector:
            details.append(f"- **Selector:** `{step.selector}`")
        if step.text:
            details.append(f"- **Text:** `{step.text}`")
        if step.key:
            details.append(f"- **Key:** `{step.key}`")
        display(Markdown("\n".join(details)))

    display(Markdown("## Execution trace"))
    for item in result.trace:
        trace_lines = [
            f"### Step {item.step_index}: `{item.action_type.value}` — `{item.status}`",
            item.description,
            f"- **Tool:** {item.tool_name or '_n/a_'}",
        ]
        if item.message:
            trace_lines.extend([
                "- **Message:**",
                f"```\n{item.message[:4000]}\n```",
            ])
        if item.snapshot_excerpt:
            trace_lines.extend([
                "- **Snapshot excerpt:**",
                f"```\n{item.snapshot_excerpt[:2000]}\n```",
            ])
        display(Markdown("\n".join(trace_lines)))

    if result.final_page is not None:
        final_lines = [
            "## Final page",
            f"- **URL:** {result.final_page.url or '_unknown_'}",
            f"- **Title:** {result.final_page.title or '_unknown_'}",
        ]
        if result.final_page.text_excerpt:
            final_lines.extend([
                "- **Visible text excerpt:**",
                f"```\n{result.final_page.text_excerpt[:4000]}\n```",
            ])
        display(Markdown("\n".join(final_lines)))

        if result.final_page.snapshot:
            display(Markdown("### Snapshot excerpt"))
            display(Markdown(f"```\n{(result.final_page.snapshot or '')[:4000]}\n```"))


display_navigation_result(result)


## Summary
- **Status:** `completed`
- **Message:** _none_
- **Planned steps:** 2
- **Executed steps:** 2
- **Available tools:** browser_click, browser_close, browser_console_messages, browser_drag, browser_drop, browser_evaluate, browser_file_upload, browser_fill_form, browser_handle_dialog, browser_hover, browser_navigate, browser_navigate_back, browser_network_requests, browser_press_key, browser_resize, browser_run_code, browser_select_option, browser_snapshot, browser_tabs, browser_take_screenshot, browser_type, browser_wait_for

## Planned steps

### Step 1: `login`
Log in as sme_admin user.

### Step 2: `snapshot`
Capture the post-login state.

## Execution trace

### Step 1: `login` — `completed`
Log in as sme_admin user.
- **Tool:** browser_navigate, browser_type, browser_type, browser_click, browser_wait_for
- **Message:**
```
### Ran Playwright code
```js
await page.goto('https://staging.acc.clearfacts.be/login');
```
### Page
- Page URL: https://staging.acc.clearfacts.be/login
- Page Title: Clearfacts voor Accountants
- Console: 0 errors, 4 warnings
### Snapshot
- [Snapshot](.playwright-mcp/page-2026-04-30T11-06-57-183Z.yml)
### Events
- New console entries: .playwright-mcp/console-2026-04-30T11-06-55-457Z.log#L1-L5
### Error
[
  {
    "expected": "string",
    "code": "invalid_type",
    "path": [
      "target"
    ],
    "message": "Invalid input: expected string, received undefined"
  }
]
### Error
[
  {
    "expected": "string",
    "code": "invalid_type",
    "path": [
      "target"
    ],
    "message": "Invalid input: expected string, received undefined"
  }
]
### Error
[
  {
    "expected": "string",
    "code": "invalid_type",
    "path": [
      "target"
    ],
    "message": "Invalid input: expected string, received undefined"
  }
]
### Error
Error: Either time, text or textGone must be provided
```
- **Snapshot excerpt:**
```
### Page
- Page URL: https://staging.acc.clearfacts.be/login
- Page Title: Clearfacts voor Accountants
- Console: 0 errors, 4 warnings
### Snapshot
```yaml
- generic [ref=e1]:
  - generic [ref=e2]:
    - img [ref=e5]
    - generic [ref=e6]:
      - heading "Aanmelden op Clearfacts NL" [level=3] [ref=e8]:
        - text: Aanmelden op Clearfacts
        - combobox [ref=e9]:
          - option "NL" [selected]
          - option "FR"
          - option "EN"
          - option "DE"
      - form [ref=e11]:
        - generic [ref=e12]:
          - generic [ref=e13]: Gebruikersnaam
          - textbox "Gebruikersnaam" [active] [ref=e14]
        - generic [ref=e15]:
          - generic [ref=e16]: Wachtwoord
          - textbox "Wachtwoord" [ref=e17]
        - generic [ref=e18]:
          - button "Aanmelden" [ref=e19] [cursor=pointer]
          - link "Wachtwoord vergeten ?" [ref=e20] [cursor=pointer]:
            - /url: /resetting/request
  - generic [ref=e22] [cursor=pointer]: 
  - generic [ref=e23]:
    - generic [ref=e24]:
      - img [ref=e25]
      - heading "staging" [level=4] [ref=e26]
    - generic [ref=e28]:
      - term [ref=e29]: Links
      - definition [ref=e30]:
        - l
```

### Step 2: `snapshot` — `completed`
Capture the post-login state.
- **Tool:** browser_snapshot
- **Message:**
```
### Page
- Page URL: https://staging.acc.clearfacts.be/login
- Page Title: Clearfacts voor Accountants
- Console: 0 errors, 4 warnings
### Snapshot
```yaml
- generic [ref=e1]:
  - generic [ref=e2]:
    - img [ref=e5]
    - generic [ref=e6]:
      - heading "Aanmelden op Clearfacts NL" [level=3] [ref=e8]:
        - text: Aanmelden op Clearfacts
        - combobox [ref=e9]:
          - option "NL" [selected]
          - option "FR"
          - option "EN"
          - option "DE"
      - form [ref=e11]:
        - generic [ref=e12]:
          - generic [ref=e13]: Gebruikersnaam
          - textbox "Gebruikersnaam" [active] [ref=e14]
        - generic [ref=e15]:
          - generic [ref=e16]: Wachtwoord
          - textbox "Wachtwoord" [ref=e17]
        - generic [ref=e18]:
          - button "Aanmelden" [ref=e19] [cursor=pointer]
          - link "Wachtwoord vergeten ?" [ref=e20] [cursor=pointer]:
            - /url: /resetting/request
  - generic [ref=e22] [cursor=pointer]: 
  - generic [ref=e23]:
    - generic [ref=e24]:
      - img [ref=e25]
      - heading "staging" [level=4] [ref=e26]
    - generic [ref=e28]:
      - term [ref=e29]: Links
      - definition [ref=e30]:
        - list [ref=e31]:
          - listitem [ref=e32]:
            - link "Jira Ticket" [ref=e33] [cursor=pointer]:
              - /url: https://clearfacts.atlassian.net/browse/staging
          - listitem [ref=e34]:
            - link "Github Branch" [ref=e35] [cursor=pointer]:
              - /url: https://github.com/Clearfacts/cf-accounting/tree/staging
          - listitem [ref=e36]:
            - link "GitHub PR" [ref=e37] [cursor=pointer]:
              - /url: https://github.com/Clearfacts/cf-accounting/pulls?q=staging
      - term [ref=e38]: Pod uptime
      - definition [ref=e39]: 10 days
    - button "Import translations" [ref=e41] [cursor=pointer]
    - generic [ref=e43]:
      - generic [ref=e44]:
        - generic [ref=e45]: Branding
        - combobox [ref=e46]:
          - option "--" [selected]
          - option "bookmate"
      - generic [ref=e47]:
        - generic [ref=e48]: Accountant
        - combobox [ref=e49]:
          - option "--" [selected]
          - option "Adsolut Accountant"
          - option "Bob 50 Accountant"
          - option "Bookmate Accountant"
          - option "Expert M Accountant"
          - option "VDL Accountant"
          - option "Winbooks Accountant"
          - option "Wings Accountant"
      - button "Apply" [ref=e50] [cursor=pointer]
      - generic [ref=e51]:
        - text: (
        - strong [ref=e52]: ESC
        - text: to close)
  - iframe [ref=e55]:
    - generic [ref=f1e5]:
      - generic [ref=f1e6]:
        - text: protected by
        - strong [ref=f1e7]: reCAPTCHA
      - generic [ref=f1e9]:
        - text: reCAPTCHA is changing its terms of service.
        - link "Take action." [ref=f1e10] [cursor=pointer]:
          - /url: https://google.com/recaptcha/admin/migrate
```
```
- **Snapshot excerpt:**
```
### Page
- Page URL: https://staging.acc.clearfacts.be/login
- Page Title: Clearfacts voor Accountants
- Console: 0 errors, 4 warnings
### Snapshot
```yaml
- generic [ref=e1]:
  - generic [ref=e2]:
    - img [ref=e5]
    - generic [ref=e6]:
      - heading "Aanmelden op Clearfacts NL" [level=3] [ref=e8]:
        - text: Aanmelden op Clearfacts
        - combobox [ref=e9]:
          - option "NL" [selected]
          - option "FR"
          - option "EN"
          - option "DE"
      - form [ref=e11]:
        - generic [ref=e12]:
          - generic [ref=e13]: Gebruikersnaam
          - textbox "Gebruikersnaam" [active] [ref=e14]
        - generic [ref=e15]:
          - generic [ref=e16]: Wachtwoord
          - textbox "Wachtwoord" [ref=e17]
        - generic [ref=e18]:
          - button "Aanmelden" [ref=e19] [cursor=pointer]
          - link "Wachtwoord vergeten ?" [ref=e20] [cursor=pointer]:
            - /url: /resetting/request
  - generic [ref=e22] [cursor=pointer]: 
  - generic [ref=e23]:
    - generic [ref=e24]:
      - img [ref=e25]
      - heading "staging" [level=4] [ref=e26]
    - generic [ref=e28]:
      - term [ref=e29]: Links
      - definition [ref=e30]:
        - l
```

## Final page
- **URL:** _unknown_
- **Title:** _unknown_
- **Visible text excerpt:**
```
### Page
- Page URL: https://staging.acc.clearfacts.be/login
- Page Title: Clearfacts voor Accountants
- Console: 0 errors, 4 warnings
### Snapshot
```yaml
- generic [ref=e1]:
  - generic [ref=e2]:
    - img [ref=e5]
    - generic [ref=e6]:
      - heading "Aanmelden op Clearfacts NL" [level=3] [ref=e8]:
        - text: Aanmelden op Clearfacts
        - combobox [ref=e9]:
          - option "NL" [selected]
          - option "FR"
          - option "EN"
          - option "DE"
      - form [ref=e11]:
        - generic [ref=e12]:
          - generic [ref=e13]: Gebruikersnaam
          - textbox "Gebruikersnaam" [active] [ref=e14]
        - generic [ref=e15]:
          - generic [ref=e16]: Wachtwoord
          - textbox "Wachtwoord" [ref=e17]
        - generic [ref=e18]:
          - button "Aanmelden" [ref=e19] [cursor=pointer]
          - link "Wachtwoord vergeten ?" [ref=e20] [cursor=pointer]:
            - /url: /resetting/request
  - generic [ref=e22] [cursor=pointer]: 
  - generic [ref=e23]:
    - generic [ref=e24]:
      - img [ref=e25]
      - heading "staging" [level=4] [ref=e26]
    - generic [ref=e28]:
      - term [ref=e29]: Links
      - definition [ref=e30]:
        - list [ref=e31]:
          - listitem [ref=e32]:
            - link "Jira Ticket" [ref=e33] [cursor=pointer]:
              - /url: https://clearfacts.atlassian.net/browse/staging
          - listitem [ref=e34]:
            - link "Github Branch" [ref=e35] [cursor=pointer]:
              - /url: https://github.com/Clearfacts/cf-accounting/tree/staging
          - listitem [ref=e36]:
            - link "GitHub PR" [ref=e37] [cursor=pointer]:
              - /url: https://github.com/Clearfacts/cf-accounting/pulls?q=staging
      - term [ref=e38]: Pod uptime
      - definition [ref=e39]: 10 days
    - button "Import translations" [ref=e41] [cursor=pointer]
    - generic [ref=e43]:
      - generic [ref=e44]:
        - generic [ref=e45]: Branding
        - combobox [ref=e46]:
          - option "--" [selected]
          - option "bookmate"
      - generic [ref=e47]:
        - generic [ref=e48]: Accountant
        - combobox [ref=e49]:
          - option "--" [selected]
          - option "Adsolut Accountant"
          - option "Bob 50 Accountant"
          - option "Bookmate Accountant"
          - option "Expert M Accountant"
          - option "VDL Accountant"
          - option "Winbooks Accountant"
          - option "Wings Accountant"
      - button "Apply" [ref=e50] [cursor=pointer]
      - generic [ref=e51]:
        - text: (
        - strong [ref=e52]: ESC
        - text: to close)
  - iframe [ref=e55]:
    - generic [ref=f1e5]:
      - generic [ref=f1e6]:
        - text: protected by
        - strong [ref=f1e7]: reCAPTCHA
      - generic [ref=f1e9]:
        - text: reCAPTCHA is changing its terms of service.
        - link "Take action." [ref=f1e10] [cursor=pointer]:
          - /url: https://google.com/recaptcha/admin/migrate
```
```

### Snapshot excerpt

```
### Page
- Page URL: https://staging.acc.clearfacts.be/login
- Page Title: Clearfacts voor Accountants
- Console: 0 errors, 4 warnings
### Snapshot
```yaml
- generic [ref=e1]:
  - generic [ref=e2]:
    - img [ref=e5]
    - generic [ref=e6]:
      - heading "Aanmelden op Clearfacts NL" [level=3] [ref=e8]:
        - text: Aanmelden op Clearfacts
        - combobox [ref=e9]:
          - option "NL" [selected]
          - option "FR"
          - option "EN"
          - option "DE"
      - form [ref=e11]:
        - generic [ref=e12]:
          - generic [ref=e13]: Gebruikersnaam
          - textbox "Gebruikersnaam" [active] [ref=e14]
        - generic [ref=e15]:
          - generic [ref=e16]: Wachtwoord
          - textbox "Wachtwoord" [ref=e17]
        - generic [ref=e18]:
          - button "Aanmelden" [ref=e19] [cursor=pointer]
          - link "Wachtwoord vergeten ?" [ref=e20] [cursor=pointer]:
            - /url: /resetting/request
  - generic [ref=e22] [cursor=pointer]: 
  - generic [ref=e23]:
    - generic [ref=e24]:
      - img [ref=e25]
      - heading "staging" [level=4] [ref=e26]
    - generic [ref=e28]:
      - term [ref=e29]: Links
      - definition [ref=e30]:
        - list [ref=e31]:
          - listitem [ref=e32]:
            - link "Jira Ticket" [ref=e33] [cursor=pointer]:
              - /url: https://clearfacts.atlassian.net/browse/staging
          - listitem [ref=e34]:
            - link "Github Branch" [ref=e35] [cursor=pointer]:
              - /url: https://github.com/Clearfacts/cf-accounting/tree/staging
          - listitem [ref=e36]:
            - link "GitHub PR" [ref=e37] [cursor=pointer]:
              - /url: https://github.com/Clearfacts/cf-accounting/pulls?q=staging
      - term [ref=e38]: Pod uptime
      - definition [ref=e39]: 10 days
    - button "Import translations" [ref=e41] [cursor=pointer]
    - generic [ref=e43]:
      - generic [ref=e44]:
        - generic [ref=e45]: Branding
        - combobox [ref=e46]:
          - option "--" [selected]
          - option "bookmate"
      - generic [ref=e47]:
        - generic [ref=e48]: Accountant
        - combobox [ref=e49]:
          - option "--" [selected]
          - option "Adsolut Accountant"
          - option "Bob 50 Accountant"
          - option "Bookmate Accountant"
          - option "Expert M Accountant"
          - option "VDL Accountant"
          - option "Winbooks Accountant"
          - option "Wings Accountant"
      - button "Apply" [ref=e50] [cursor=pointer]
      - generic [ref=e51]:
        - text: (
        - strong [ref=e52]: ESC
        - text: to close)
  - iframe [ref=e55]:
    - generic [ref=f1e5]:
      - generic [ref=f1e6]:
        - text: protected by
        - strong [ref=f1e7]: reCAPTCHA
      - generic [ref=f1e9]:
        - text: reCAPTCHA is changing its terms of service.
        - link "Take action." [ref=f1e10] [cursor=pointer]:
          - /url: https://google.com/recaptcha/admin/migrate
```
```

In [6]:
# Optional: inspect the raw payload if you need it for debugging.
raw_payload = result.model_dump(mode="json")
print(json.dumps(raw_payload, indent=2)[:8000])


{
  "status": "completed",
  "request": {
    "instruction": "Log in as an SME user",
    "role": "sme_admin",
    "start_target": null,
    "max_steps": 8,
    "include_snapshot": true
  },
  "plan": {
    "supported": true,
    "message": null,
    "steps": [
      {
        "action_type": "login",
        "description": "Log in to ClearFacts as SME admin (sme_admin).",
        "target": null,
        "selector": null,
        "text": null,
        "key": null
      },
      {
        "action_type": "snapshot",
        "description": "Capture a snapshot after successful login.",
        "target": null,
        "selector": null,
        "text": null,
        "key": null
      }
    ]
  },
  "trace": [
    {
      "step_index": 1,
      "action_type": "login",
      "description": "Log in to ClearFacts as SME admin (sme_admin).",
      "tool_name": "browser_navigate, browser_type, browser_type, browser_click, browser_wait_for",
      "arguments": {
        "calls": [
          {
      